In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

# All directories are configured here. Hide this cell to hide local paths.
from pathlib import Path
DATA_DIR = PROCESSED_DIR
REPAIR_DIR = DATA_DIR / 'repair_2024_2025'
ZONE_DIR = ZONE_DIR
RESULTS_DIR = ANALYSIS_DIR
YEARS = [2019, 2021, 2022, 2023, 2024, 2025]
# Read validated repair candidates in place; never promote or overwrite originals.
USE_VALIDATED_REPAIRS = True
SAVE_FIGURES = True


# Baseline pickup and drop-off demand

## Scope and interpretation

These notebooks analyze **cleaned trip totals**, not trip duration, raw requests,
vehicle availability, or a causal policy effect. Combined tables do not identify platforms.
2019 includes all recorded HVFHV platforms; do not label the series Uber-only.
January 2019 and all of 2020 are excluded. Annual figures describe available months;
use common-month comparisons for cross-year conclusions.

Pickup tables use pickup calendar date/hour. Drop-off tables for 2019/2021–2023
use actual drop-off calendar date/hour. The 2024–2025 drop-off tables instead use
pickup date with drop-off hour. Consequently, they describe destination activity
for a pickup-date cohort. February 2019 and January 2021 drop-offs have missing
prior-month spill-in and are boundary-limited despite full calendar-date coverage.
Hour labels are local clock labels; repeated daylight-saving hours are combined.
Absent zone-hours are treated as zero only within included, complete months.

Validated staged drop-off repairs may be read in place using the configuration
flag. This does not replace any original analytical output. All directories are
defined in the first code cell. Dependencies: pandas, numpy, matplotlib, duckdb;
the spatial notebook also needs geopandas and its shapefile-reading dependencies.



In [ ]:
import calendar
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
db = duckdb.connect()
db.execute("SET threads=1")
db.execute("SET memory_limit='256MB'")
db.execute("SET preserve_insertion_order=false")
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 110})
def finish(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(RESULTS_DIR / (name + '.png'), dpi=180, bbox_inches='tight')
    plt.show()


In [ ]:
NOTEBOOK_ID = '03_baseline'
DIRECTIONS = ['pickup', 'dropoff']


In [ ]:
# Validate directory selection and key/date/count integrity before analysis.
stems = {'pickup': 'fhvhv_by_zone_hour', 'dropoff': 'fhvhv_by_DOzone_hour'}
repair_tasks = {('dropoff', 2025, 8), ('dropoff', 2025, 10)}
manifest, daily_frames, zone_frames, profile_frames = [], [], [], []
for year in YEARS:
    for month in range(2 if year == 2019 else 1, 13):
        for direction, stem in stems.items():
            if direction not in DIRECTIONS:
                continue
            item = dict(direction=direction, year=year, month=month)
            folder = DATA_DIR / f'{stem}_{year}_{month:02d}.parquet'
            if USE_VALIDATED_REPAIRS and (direction, year, month) in repair_tasks:
                candidate = REPAIR_DIR / folder.name
                marker = candidate / '_repair_validation.json'
                if marker.exists():
                    record = json.loads(marker.read_text(encoding='utf-8'))
                    contract = record['contract']
                    assert record['passed'] and contract['version'] == 'selected-repairs-v1-legacy-dropoff-date'
                    assert (contract['family'], contract['year'], contract['month']) == (direction, year, month)
                    folder = candidate
            if (direction, year, month) in repair_tasks and folder.parent == DATA_DIR:
                item.update(status='excluded_unrepaired', path=str(folder))
                manifest.append(item)
                continue
            parts = sorted(folder.glob('*.parquet'))
            if not parts or not (folder / '_SUCCESS').exists():
                item.update(status='missing_or_incomplete', path=str(folder))
                manifest.append(item)
                continue
            if year in [2019, 2021, 2022, 2023]:
                contract = json.loads((folder / '_aggregation_contract.json').read_text(encoding='utf-8'))
                assert contract['version'] == 'earlier-years-hourly-v2'
                assert (contract['family'], contract['year'], contract['month']) == (direction, year, month)
            zone = 'PULocationID' if direction == 'pickup' else 'DOLocationID'
            hour = 'hour' if direction == 'pickup' else 'do_hour'
            db.read_parquet([str(p) for p in parts]).create_view('source', replace=True)
            bad = db.execute(f"""SELECT count(*) FROM source WHERE
                "{zone}" IS NULL OR date IS NULL OR "{hour}" IS NULL
                OR "{hour}" NOT BETWEEN 0 AND 23 OR trip_count IS NULL OR trip_count<=0
                OR year(date)!={year} OR month(date)!={month}""").fetchone()[0]
            duplicates = db.execute(f"""SELECT count(*) FROM
                (SELECT 1 FROM source GROUP BY "{zone}",date,"{hour}" HAVING count(*)>1)""").fetchone()[0]
            dates = db.execute('SELECT count(DISTINCT date) FROM source').fetchone()[0]
            assert bad == duplicates == 0, f'Invalid rows/duplicate keys: {folder.name}'
            assert dates == calendar.monthrange(year, month)[1], f'Incomplete dates: {folder.name}'
            time_basis = ('pickup_date' if direction == 'pickup' else
                          'actual_dropoff_date' if year < 2024 else 'pickup_date_with_dropoff_hour')
            item.update(status='included', path=str(folder), time_basis=time_basis,
                        boundary_limited=direction=='dropoff' and (year,month) in [(2019,2),(2021,1)])
            manifest.append(item)
            for frames, sql in [
                (daily_frames, 'SELECT date, sum(trip_count) trips FROM source GROUP BY date'),
                (zone_frames, f'SELECT "{zone}" zone_id, sum(trip_count) trips FROM source GROUP BY "{zone}"'),
                (profile_frames, f'SELECT dayofweek(date) AS dow, "{hour}" AS "hour", sum(trip_count) AS trips FROM source GROUP BY dayofweek(date),"{hour}"')]:
                frame = db.execute(sql).df()
                for key, value in [('direction',direction),('year',year),('month',month),('time_basis',time_basis)]:
                    frame[key] = value
                frames.append(frame)

manifest = pd.DataFrame(manifest)
display(manifest.groupby(['direction','year','status']).size().rename('months').to_frame())
display(manifest[manifest.status != 'included'])
assert daily_frames, 'No complete inputs selected'
daily = pd.concat(daily_frames, ignore_index=True)
daily['date'] = pd.to_datetime(daily.date)
zone_month = pd.concat(zone_frames, ignore_index=True)
profiles = pd.concat(profile_frames, ignore_index=True)
manifest.to_csv(RESULTS_DIR / f'{NOTEBOOK_ID}_input_manifest.csv', index=False)
daily.to_csv(RESULTS_DIR / f'{NOTEBOOK_ID}_daily_totals.csv', index=False)
print('Only included months enter plots. Missing months are not zero demand.')


## Hourly demand
Separate panels preserve the different hourly definitions. The rolling mean requires seven observed days.

In [ ]:
# Uses DATA_DIR, REPAIR_DIR, and manifest from the notebook.
# Run after the data-loading cell.

hourly_frames = []

for item in manifest.loc[manifest.status.eq('included')].itertuples():
    folder = Path(item.path)
    hour_col = 'hour' if item.direction == 'pickup' else 'do_hour'

    db.read_parquet(
        [str(p) for p in folder.glob('*.parquet')]
    ).create_view('hourly_source', replace=True)

    frame = db.execute(f'''
        SELECT date, "{hour_col}" AS "hour",
               SUM(trip_count) AS trips
        FROM hourly_source
        GROUP BY date, "{hour_col}"
    ''').df()

    frame['direction'] = item.direction
    hourly_frames.append(frame)

hourly = pd.concat(hourly_frames, ignore_index=True)
hourly['timestamp'] = (
    pd.to_datetime(hourly['date'])
    + pd.to_timedelta(hourly['hour'], unit='h')
)

fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)

for ax, direction in zip(axes, ['pickup', 'dropoff']):
    s = (
        hourly.loc[hourly.direction.eq(direction)]
        .set_index('timestamp')['trips']
        .sort_index()
        .asfreq('h')
    )

    ax.plot(
        s.index, s / 1e3,
        color='#444444', lw=0.35, alpha=0.65,
        label='Hourly trips'
    )
    ax.plot(
        s.index, s.rolling(168, min_periods=168).mean() / 1e3,
        color='#D55E00', lw=1.5,
        label='7-day rolling mean (168 hours)'
    )

    ax.set(
        title=direction.title(),
        ylabel='Thousand trips/hour'
    )
    ax.legend(loc='upper left')

axes[-1].set_xlabel('Date')
fig.suptitle('Hourly Pickup and Drop-off Demand', fontsize=18)
finish(fig, '03_hourly_demand_entire_period')

## Daily demand
Separate panels preserve the different date definitions. The rolling mean requires seven observed days.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, direction in zip(axes, DIRECTIONS):
    s = (
        daily[daily.direction == direction]
        .set_index("date").trips.sort_index().asfreq("D")
    )

    ax.plot(
        s.index, s / 1e3,
        color="black", alpha=0.7, linewidth=1,
        label="Daily", zorder=1
    )
    ax.plot(
        s.index, s.rolling(7, min_periods=7).mean() / 1e3,
        color="#D55E00", alpha=1, linewidth=2,
        label="7-day mean", zorder=2
    )

    ax.set(title=direction.title(), ylabel="Thousand trips/day")
    ax.legend()

fig.suptitle("Daily Pickup and Drop-off Demand", fontsize=16)
finish(fig, "03_daily_demand")

In [ ]:
zero_pickups = daily.loc[(daily["direction"] == "pickup") & (daily["trips"] <200000)].sort_values("date")

display(zero_pickups)

## Monthly demand and calendar exposure
Show both totals and trips per observed calendar day. Gaps remain gaps.

In [ ]:
monthly = daily.groupby(['direction','year','month'],as_index=False).agg(trips=('trips','sum'), days=('date','nunique'))
monthly['trips_per_day'] = monthly.trips / monthly.days
fig, axes = plt.subplots(1,2,figsize=(15,5))
for direction in DIRECTIONS:
    m = monthly[monthly.direction==direction].copy()
    m['date'] = pd.to_datetime(dict(year=m.year,month=m.month,day=1))
    start, end = m.date.min(), m.date.max()
    m = m.set_index('date').reindex(pd.date_range(start,end,freq='MS'))
    axes[0].plot(m.index,m.trips/1e6,label=direction)
    axes[1].plot(m.index,m.trips_per_day/1e3,label=direction)
axes[0].set(title='Monthly totals',ylabel='Million trips')
axes[1].set(title='Calendar-day normalized demand',ylabel='Thousand trips/day')
for ax in axes: ax.legend()
finish(fig,'03_monthly_demand')
display(monthly)
monthly.to_csv(RESULTS_DIR/'03_monthly_totals.csv',index=False)


## Comparable-month year summaries
Use months present in every selected year and both directions. This avoids comparing a partial year with a full year.

In [ ]:
coverage = monthly.groupby(['direction','year']).month.apply(set)
common_months = sorted(set.intersection(*coverage.tolist()))
comparison = monthly[monthly.month.isin(common_months)].groupby(['direction','year']).agg(trips=('trips','sum'),days=('days','sum'))
comparison['trips_per_day'] = comparison.trips/comparison.days
print('Common months:',common_months)
display(comparison)
fig,ax=plt.subplots()
comparison.trips_per_day.unstack('direction').div(1e3).plot.bar(ax=ax)
ax.set(ylabel='Thousand trips/day',title='Demand in common observed months')
finish(fig,'03_common_month_year_comparison')


## Recorded findings and data-quality decisions

### Findings from the hourly demand plots

- Hourly trip counts show strong recurring variation, with regular peaks and overnight lows.
- Recorded demand exhibits seasonal variation, tending to weaken during summer and strengthen toward year-end. Subsequent analyses will examine daily and hourly patterns in greater detail.
- Recorded demand rises from early 2021, followed by a more gradual increase through 2025. These are descriptive trends, not causal findings.
- Pickup and drop-off trends broadly track each other. Counts in the same hour need not match because trips start and finish at different times.
- The blank interval in 2020 reflects excluded data, not zero demand. January 2019 is also outside the analysis scope.

### Data issues and treatment

| Issue | Evidence and interpretation | Decision |
|---|---|---|
| Six anomalous dates in 2019 | July 31; August 1–4 and August 19. Raw trip counts were normal, but financial cleaning filters removed unusually large numbers of Uber records. | Use hourly imputation for these six dates; no raw-data reaggregation. |
| Broad hourly suppression | All six dates contain 24 nonzero hourly totals. July 31 and August 1–4 have daily totals around 24% of other matching-weekday means within the month. August 19 is around 31%, with some hourly recovery. Drop-offs show similar patterns. | For each affected hour, use the mean for the same hour and weekday within that calendar month, separately for pickups and drop-offs. Exclude all six affected dates from the donor observations. |
| Other unusually low hours | Other dips are visible, but their causes have not been verified. Low demand alone does not establish a data error. | Preserve these observations. Do not use an arbitrary ratio cutoff to select replacements. |

<!-- ### Hourly analysis and modeling record -->

<!-- Hourly demand is the intended modeling target; daily totals are descriptive only. The agreed correction is an analysis-layer estimate of typical hourly demand, not a recovery of the missing observations. Preserve original counts as `trips_observed`, adjusted counts as `trips`, and an `imputed` flag. Original analytical Parquet files must remain unchanged. -->

<!-- For future model evaluation, exclude imputed hours from scoring and construct imputation references using training observations only, without borrowing from the evaluation period. -->

<!-- The plots describe **cleaned, recorded trips across available HVFHV platforms**, not unmet demand or an Uber-only series. Different hourly pickup and drop-off counts are expected and do not by themselves indicate a data problem. -->
